In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

In [3]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
)

CURATED_DIR = (
    PROJECT_DIR
    / "data"
    / "curated"
)

# Create curated folder if it does not exist
CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
customers_df = pd.read_csv(
    RAW_DIR / "customers.csv"
)

policies_df = pd.read_csv(
    RAW_DIR / "policies.csv"
)

In [5]:
payment_features = pd.read_csv(
    PROCESSED_DIR /
    "policy_payment_features.csv"
)

claim_features = pd.read_csv(
    PROCESSED_DIR /
    "policy_claim_features.csv"
)

underwriting_features = pd.read_csv(
    PROCESSED_DIR /
    "policy_underwriting_features.csv"
)

renewal_features = pd.read_csv(
    PROCESSED_DIR /
    "policy_renewal_features.csv"
)

In [6]:
renewals_raw = pd.read_csv(
    RAW_DIR / "renewals.csv"
)

In [7]:
print("Customers    :", customers_df.shape)
print("Policies     :", policies_df.shape)

print(
    "Payments     :",
    payment_features.shape
)

print(
    "Claims       :",
    claim_features.shape
)

print(
    "Underwriting :",
    underwriting_features.shape
)

print(
    "Renewals     :",
    renewal_features.shape
)

Customers    : (100000, 12)
Policies     : (150000, 15)
Payments     : (73102, 12)
Claims       : (58905, 22)
Underwriting : (50000, 16)
Renewals     : (19204, 12)


Why different sizes?

Because not every policy appears in every transactional/source table.

In [8]:
print("Policy duplicate:", policies_df["POLICY_ID"].duplicated().sum())
print("Payment duplicate:", payment_features["POLICY_ID"].duplicated().sum())
print("Claim duplicate:", claim_features["POLICY_ID"].duplicated().sum())
print("Underwriting duplicate:", underwriting_features["POLICY_ID"].duplicated().sum())
print("Renewal duplicate:", renewal_features["POLICY_ID"].duplicated().sum())

Policy duplicate: 0
Payment duplicate: 0
Claim duplicate: 0
Underwriting duplicate: 0
Renewal duplicate: 0


Start With Policies as Base Table

In [9]:
policy_360 = policies_df.copy()

Create Merge Validation Function

In [10]:
BASE_POLICY_COUNT = len(policies_df)


def validate_policy_360(df, step):
    """
    Validate Policy 360 after each merge.
    """

    print(f"{step}: {df.shape}")

    # Row count must remain unchanged
    assert len(df) == BASE_POLICY_COUNT, (f"Row count changed after {step}")

    # Policy ID must remain unique
    assert df["POLICY_ID"].is_unique, (f"Duplicate POLICY_ID after {step}")

assert means:

I expect this condition to be True.

If it isn't:

Python immediately stops

Add Customer Information

In [11]:
customer_cols = [
    "CUSTOMER_ID",
    "AGE",
    "GENDER",
    "MARITAL_STATUS",
    "OCCUPATION",
    "ANNUAL_INCOME",
    "STATE",
    "CITY",
    "CREDIT_SCORE",
    "CUSTOMER_RISK_SEGMENT",
    "CUSTOMER_SINCE"
]

In [12]:
policy_360 = policy_360.merge(customers_df[customer_cols],
    on="CUSTOMER_ID",
    how="left",
    validate="many_to_one"
)

In [13]:
validate_policy_360(policy_360,"Customer Merge")

Customer Merge: (150000, 25)


In [14]:
payment_features[
    "HAS_PAYMENT_RECORD"
] = 1

In [15]:
policy_360 = policy_360.merge(
    payment_features,
    on=[
        "POLICY_ID",
        "CUSTOMER_ID"
    ],
    how="left",
    validate="one_to_one"
)

In [16]:
validate_policy_360(
    policy_360,
    "Payment Merge"
)

Payment Merge: (150000, 36)


In [17]:
claim_features[
    "HAS_CLAIM_RECORD"
] = 1

In [18]:
policy_360 = policy_360.merge(
    claim_features,
    on=[
        "POLICY_ID",
        "CUSTOMER_ID"
    ],
    how="left",
    validate="one_to_one"
)

In [19]:
validate_policy_360(
    policy_360,
    "Claim Merge"
)

Claim Merge: (150000, 57)


Rename Underwriting Columns First

In [20]:
underwriting_features = underwriting_features.rename(
    columns={
        "AGE": "UW_AGE",
        "CREDIT_SCORE": "UW_CREDIT_SCORE",
        "DECISION": "UW_DECISION"
    }
)

In [21]:
underwriting_features[
    "HAS_UNDERWRITING_RECORD"
] = 1

In [22]:
policy_360 = policy_360.merge(underwriting_features,on=["POLICY_ID","CUSTOMER_ID"],
    how="left",
    validate="one_to_one"
)

In [23]:
validate_policy_360(
    policy_360,
    "Underwriting Merge"
)

Underwriting Merge: (150000, 72)


In [24]:
renewal_source_flag = (renewals_raw[["POLICY_ID"]] .drop_duplicates()  .copy()
)

renewal_source_flag[
    "HAS_RENEWAL_RECORD"
] = 1

In [25]:
policy_360 = policy_360.merge(renewal_source_flag,
    on="POLICY_ID",
    how="left",
    validate="one_to_one"
)

In [26]:
validate_policy_360(
    policy_360,
    "Renewal Source Flag Merge"
)

Renewal Source Flag Merge: (150000, 73)


In [27]:
renewal_features[
    "HAS_FINALIZED_RENEWAL"
] = 1

In [28]:
policy_360 = policy_360.merge(
    renewal_features,
    on=[
        "POLICY_ID",
        "CUSTOMER_ID"
    ],
    how="left",
    validate="one_to_one"
)

In [29]:
validate_policy_360(
    policy_360,
    "Renewal Feature Merge"
)

Renewal Feature Merge: (150000, 84)


In [30]:
policy_360.shape

(150000, 84)

In [31]:
policy_360[
    "POLICY_ID"
].nunique()

150000

In [32]:
policy_360[
    "POLICY_ID"
].duplicated().sum()

np.int64(0)

In [33]:
presence_cols = [
    "HAS_PAYMENT_RECORD",
    "HAS_CLAIM_RECORD",
    "HAS_UNDERWRITING_RECORD",
    "HAS_RENEWAL_RECORD",
    "HAS_FINALIZED_RENEWAL"
]

policy_360[presence_cols] = policy_360[presence_cols].fillna(0).astype(int)

In [34]:
coverage_summary = policy_360[presence_cols].sum().to_frame("POLICIES_WITH_RECORD")

coverage_summary["COVERAGE_PCT"] = (
    coverage_summary["POLICIES_WITH_RECORD"] / len(policy_360) * 100
)

coverage_summary.round(2)

,POLICIES_WITH_RECORD,COVERAGE_PCT
HAS_PAYMENT_RECORD,73102,48.73
HAS_CLAIM_RECORD,58905,39.27
HAS_UNDERWRITING_RECORD,50000,33.33
HAS_RENEWAL_RECORD,20000,13.33
HAS_FINALIZED_RENEWAL,19204,12.80


In [35]:
claim_zero_cols = [
    "TOTAL_CLAIMS",
    "TOTAL_CLAIM_AMOUNT",
    "TOTAL_SETTLEMENT_AMOUNT",
    "APPROVED_CLAIM_COUNT",
    "PARTIAL_APPROVED_CLAIM_COUNT",
    "REJECTED_CLAIM_COUNT",
    "PENDING_CLAIM_COUNT",
    "UNDER_INVESTIGATION_COUNT",
    "SUSPECTED_FRAUD_COUNT",
    "CONFIRMED_FRAUD_COUNT",
    "HIGH_SEVERITY_CLAIM_COUNT",
    "HAS_FRAUD_ALERT"
]

In [36]:
policy_360[claim_zero_cols] = (
    policy_360[claim_zero_cols
    ].fillna(0)
)

In [37]:
payment_zero_cols = [
    "TOTAL_PAYMENT_TRANSACTIONS",
    "TOTAL_PREMIUM_AMOUNT",
    "PAID_PAYMENT_COUNT",
    "FAILED_PAYMENT_COUNT",
    "OVERDUE_PAYMENT_COUNT",
    "PENDING_PAYMENT_COUNT",
    "TOTAL_CHANNEL_COST"
]

In [38]:
policy_360[payment_zero_cols] = (
    policy_360[payment_zero_cols
    ].fillna(0)
)

In [39]:
date_cols = [
    "POLICY_START_DATE",
    "POLICY_END_DATE",
    "CUSTOMER_SINCE",
    "LAST_CLAIM_DATE"
]

for col in date_cols:
    if col in policy_360.columns:
        policy_360[col] = pd.to_datetime(policy_360[col], errors="coerce")

In [40]:
policy_360["POLICY_TERM_DAYS"] = (
    policy_360["POLICY_END_DATE"] - policy_360["POLICY_START_DATE"]
).dt.days

In [41]:
policy_360[
    "PREMIUM_TO_SUM_INSURED_PCT"
] = (
    policy_360[
        "ANNUAL_PREMIUM"
    ]
    /
    policy_360[
        "SUM_INSURED"
    ]
    * 100
)

In [42]:
# Annual Premium = ₹20,000
# Sum Insured    = ₹10,00,000

# 20,000 / 10,00,000 × 100

# = 2%

# This is not an actuarial profitability metric.

# It's simply a premium-to-coverage ratio.

In [43]:
policy_360[
    "ANY_CLAIM_FLAG"
] = (
    policy_360[
        "TOTAL_CLAIMS"
    ] > 0
).astype(int)

In [44]:
policy_360["PAYMENT_PROBLEM_FLAG"] = (
    (policy_360["FAILED_PAYMENT_COUNT"] > 0) |
    (policy_360["OVERDUE_PAYMENT_COUNT"] > 0)
).astype(int)

In [45]:
policy_360[
    "AGE"
].isna().sum()

np.int64(0)

In [46]:
policy_360[
    "CUSTOMER_RISK_SEGMENT"
].isna().sum()

np.int64(0)

In [47]:
uw_age_check = policy_360[
    policy_360["HAS_UNDERWRITING_RECORD"] == 1
]

In [48]:
(uw_age_check["AGE"] != uw_age_check["UW_AGE"]).sum()

np.int64(0)

Create Final Policy 360 Quality Summary

In [49]:
policy_360_summary = pd.DataFrame({
    
    "METRIC": [
        "Rows",
        "Unique Policies",
        "Unique Customers",
        "Policies with Payment Data",
        "Policies with Claims",
        "Policies with Underwriting",
        "Policies with Renewal Record",
        "Policies with Finalized Renewal"
    ],

    "VALUE": [
        len(policy_360),
        policy_360["POLICY_ID"].nunique(),
        policy_360["CUSTOMER_ID"].nunique(),
        policy_360["HAS_PAYMENT_RECORD"].sum(),
        policy_360["HAS_CLAIM_RECORD"].sum(),
        policy_360["HAS_UNDERWRITING_RECORD"].sum(),
        policy_360["HAS_RENEWAL_RECORD"].sum(),
        policy_360["HAS_FINALIZED_RENEWAL"].sum()
    ]
})

policy_360_summary

,METRIC,VALUE
0,Rows,150000
1,Unique Policies,150000
2,Unique Customers,77891
3,Policies with Payment Data,73102
4,Policies with Claims,58905
5,Policies with Underwriting,50000
6,Policies with Renewal Record,20000
7,Policies with Finalized Renewal,19204


Check Target Distribution

In [50]:
policy_360[
    "RENEWED_FLAG"
].value_counts(
    dropna=False
)

RENEWED_FLAG
NaN    130796
1.0     17334
0.0      1870
Name: count, dtype: int64

In [51]:
policy_360.to_csv(
    CURATED_DIR /
    "policy_360.csv",
    index=False
)

In [52]:
policy_360_check = pd.read_csv(
    CURATED_DIR /
    "policy_360.csv"
)

print(
    policy_360_check.shape
)

print(
    policy_360_check[
        "POLICY_ID"
    ].nunique()
)

(150000, 88)
150000


In [55]:
policy_360_check.head(10)

,POLICY_ID,CUSTOMER_ID,POLICY_TYPE,SALES_CHANNEL,AGENT_ID,POLICY_START_DATE,POLICY_END_DATE,SUM_INSURED,ANNUAL_PREMIUM,PAYMENT_MODE,RISK_SCORE,RISK_BAND,POLICY_STATUS,DISCOUNT_PCT,COMMISSION_PCT,AGE,GENDER,MARITAL_STATUS,OCCUPATION,ANNUAL_INCOME,STATE,CITY,CREDIT_SCORE,CUSTOMER_RISK_SEGMENT,CUSTOMER_SINCE,TOTAL_PAYMENT_TRANSACTIONS,TOTAL_PREMIUM_AMOUNT,PAID_PAYMENT_COUNT,FAILED_PAYMENT_COUNT,OVERDUE_PAYMENT_COUNT,PENDING_PAYMENT_COUNT,AVG_PAYMENT_DELAY,MAX_PAYMENT_DELAY,TOTAL_CHANNEL_COST,PAYMENT_SUCCESS_RATE,HAS_PAYMENT_RECORD,TOTAL_CLAIMS,TOTAL_CLAIM_AMOUNT,TOTAL_SETTLEMENT_AMOUNT,AVG_CLAIM_AMOUNT,MAX_CLAIM_AMOUNT,AVG_PROCESSING_DAYS,MAX_PROCESSING_DAYS,AVG_REPORTING_DELAY_DAYS,APPROVED_CLAIM_COUNT,PARTIAL_APPROVED_CLAIM_COUNT,REJECTED_CLAIM_COUNT,PENDING_CLAIM_COUNT,UNDER_INVESTIGATION_COUNT,SUSPECTED_FRAUD_COUNT,CONFIRMED_FRAUD_COUNT,HIGH_SEVERITY_CLAIM_COUNT,AVG_FRAUD_SCORE,LAST_CLAIM_DATE,CLAIM_SETTLEMENT_RATIO,HAS_FRAUD_ALERT,HAS_CLAIM_RECORD,UW_AGE,HEALTH_SCORE,LIFESTYLE,MEDICAL_HISTORY_FLAG,SMOKER_FLAG,BMI,OCCUPATION_RISK,UW_CREDIT_SCORE,UNDERWRITING_SCORE,UW_DECISION,PREMIUM_LOADING_PCT,UW_APPROVED_FLAG,UW_LOADING_FLAG,UW_DECLINED_FLAG,HAS_UNDERWRITING_RECORD,HAS_RENEWAL_RECORD,OLD_PREMIUM,NEW_PREMIUM,PREMIUM_INCREASE_PCT,CLAIM_HISTORY_FLAG,LOYALTY_YEARS,CONTACT_CHANNEL,RENEWAL_STATUS,RENEWED_FLAG,PREMIUM_CHANGE_AMOUNT,PREMIUM_INCREASED_FLAG,HAS_FINALIZED_RENEWAL,POLICY_TERM_DAYS,PREMIUM_TO_SUM_INSURED_PCT,ANY_CLAIM_FLAG,PAYMENT_PROBLEM_FLAG
0,POL000000001,CUST00047547,Motor,Corporate Partner,AGT004657,2022-06-14,2042-06-09,485748,11289,Annual,434,Low,Active,11.93,4.53,38,Male,Single,Business Owner,297309,Karnataka,Mysuru,790,High,2023-02-09,1.0,940.0,1.0,0.0,0.0,0.0,20.0,20.0,27.98,1.0,1,1.0,56858.0,0.0,56858.0,56858.0,16.0,16.0,6.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.51210,2022-07-06,0.000000,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,7300,2.324045,1,0
1,POL000000002,CUST00065458,Motor,Corporate Partner,AGT000403,2020-10-02,2021-10-02,496150,13822,Monthly,284,Low,Active,10.27,2.41,31,Female,Single,Salaried,1118290,Maharashtra,Nagpur,712,Medium,2019-07-15,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.00,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0,31.0,76.0,Average,No,No,22.3,Medium,712.0,606.0,Approved,0.00,1.0,0.0,0.0,1,1,13822.0,14110.0,2.09,Yes,4.0,SMS,Renewed,1.0,288.0,1.0,1,365,2.785851,0,0
2,POL000000003,CUST00074570,Term Life,Agent,AGT003370,2023-12-30,2028-12-28,8290365,17205,Annual,391,Low,Active,10.42,3.57,38,Male,Married,Self Employed,1243028,Maharashtra,Nagpur,747,Medium,2020-01-29,2.0,34410.0,2.0,0.0,0.0,0.0,2.5,5.0,908.78,1.0,1,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0,38.0,81.0,Good,No,Yes,21.3,High,747.0,703.0,Approved,0.00,1.0,0.0,0.0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1825,0.207530,0,0
3,POL000000004,CUST00043802,Motor,Bancassurance,AGT003064,2021-01-26,2022-01-26,321823,26296,Monthly,360,Low,Active,13.19,4.92,42,Other,Single,Self Employed,795284,Rajasthan,Udaipur,778,Low,2022-08-25,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.00,NaN,0,1.0,32445.0,32429.0,32445.0,32445.0,15.0,15.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.06510,2026-08-22,0.999507,0.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,365,8.170951,1,0
4,POL000000005,CUST00015974,Commercial,Broker,AGT003426,2023-05-21,2024-05-20,7695457,24541,Monthly,345,Low,Active,10.85,3.05,20,Other,Married,Professional,1385946,Telangana,Warangal,876,Low,2018-01-18,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.00,NaN,0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0,20.0,76.0,Good,No,No,17.2,Low,876.0,751.0,Approved,0.00,1.0,0.0,0.0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,365,0.318902,0,0
5,POL000000006,CUST00095870,Home,Broker,AGT002977,2023-08-15,2024-08-14,2221956,15516,Annual,492,Medium,Cancelled,19.80,0.50,48,Male,Single,Salaried,4135